## Exercise — Loan-Approval Audit (XAI Memo on 2 Flagged Cases)

You are the AI Risk Officer at UdaciBank. The new loan-approval model has generated consumer complaints from declined applicants. Two cases are on your desk for an audit memo that the model-risk + legal teams will rely on.

**Data:**
- `data/loan_classifier.joblib` — the production gradient-boosting model
- `data/loan_test.csv` — the held-out test set
- `data/flagged_cases.csv` — the two flagged declined applicants

**Deliverable (this notebook):**
1. SHAP per-case waterfall plots saved as `shap_explanations_case1.png` + `shap_explanations_case2.png`.
2. Global SHAP feature-importance bar chart saved as `shap_global_importance.png`.
3. A `detect_proxy_features()` helper that flags features acting as proxies for protected attributes.
4. The audit memo embedded as a markdown cell at the end of the notebook.

In [ ]:
import pandas as pd
import numpy as np
import joblib
import shap
import matplotlib.pyplot as plt

bundle = joblib.load("data/loan_classifier.joblib")
clf = bundle["model"]
feature_names = bundle["feature_names"]

test    = pd.read_csv("data/loan_test.csv")
flagged = pd.read_csv("data/flagged_cases.csv")
print(f"Loaded model + {len(feature_names)} features. Test rows = {len(test)}; Flagged cases = {len(flagged)}")
flagged

## 1. SHAP for the two flagged cases

In [ ]:
# TODO Step 1: build a TreeExplainer over the test set as background. For sklearn GBM use:
#     explainer = shap.TreeExplainer(
#         clf, data=test[feature_names].values, feature_perturbation="interventional"
#     )
# TODO Step 2: shap_values_test    = explainer.shap_values(test[feature_names].values,    check_additivity=False)
# TODO Step 3: shap_values_flagged = explainer.shap_values(flagged[feature_names].values, check_additivity=False)
# TODO Step 4: print, for each flagged case:
#   - the model's predicted approval probability (clf.predict_proba(...)[:, 1])
#   - the top 5 features by |SHAP value| with sign
pass

## 2. Save per-case SHAP waterfall plots

In [ ]:
# TODO Step 5: render and save shap_explanations_case1.png and shap_explanations_case2.png
# Hint:
#   exp = shap.Explanation(
#       values        = shap_values_flagged[i],
#       base_values   = explainer.expected_value,
#       data          = flagged[feature_names].iloc[i].values,
#       feature_names = feature_names,
#   )
#   shap.plots.waterfall(exp, show=False)
#   plt.savefig(f"shap_explanations_case{i+1}.png", dpi=150, bbox_inches="tight")
#   plt.close()
pass

## 3. Global feature importance

In [ ]:
# TODO Step 6: render and save shap_global_importance.png as a horizontal bar chart of
#   global mean |SHAP| per feature, sorted descending.
pass

## 4. Detect proxy features

In [ ]:
def detect_proxy_features(local_shap_matrix, feature_names, suspect_features, multiplier=2.0):
    """Flag features whose mean local-impact across the flagged cases is materially larger
    than the model's global mean impact in the same direction.

    Args:
        local_shap_matrix: shap_values for ONLY the flagged cases (n_flagged, n_features).
        feature_names:     list of feature names (matches matrix columns).
        suspect_features:  list of feature names that are candidate proxies (e.g., zip_income_index,
                           employer_size_score, education_score).
        multiplier:        flag a feature when local_mean_abs > multiplier * global_mean_abs.

    Returns:
        DataFrame with columns: feature, local_mean, global_mean, ratio, flagged_as_proxy.
    """
    # TODO Step 7: compute global_mean_abs per feature from your full-test SHAP matrix
    # TODO Step 8: compute local_mean_abs per feature from local_shap_matrix (the 2 flagged cases)
    # TODO Step 9: return the per-feature DataFrame and mark feature as flagged where:
    #   feature is in suspect_features AND local_mean_abs > multiplier * global_mean_abs
    pass

SUSPECT = ["zip_income_index", "employer_size_score", "education_score"]
# proxy_report = detect_proxy_features(shap_values_flagged, feature_names, SUSPECT)
# print(proxy_report)

## 5. Feature-removal counterfactual (the analytical step the demo doesn't cover)

The demo computed SHAP attribution only. Here you'll go one step further: re-score the two flagged cases with the most-suspect proxy feature ablated (set to a neutral value), and quantify how much approval probability and SHAP attribution change. This tells the AI Risk Officer whether the proxy was *actually* driving the decline or whether it was correlated with another driver.

In [ ]:
# TODO Step 10: from your proxy_report, pick the most-suspect feature
#   (highest ratio among rows where flagged_as_proxy == True).
# top_proxy = proxy_report[proxy_report["flagged_as_proxy"]].iloc[0]["feature"]

# TODO Step 11: ablate that feature by setting it to its training-set median (a neutral baseline).
# median_value = float(test[top_proxy].median())
# flagged_ablated = flagged.copy()
# flagged_ablated[top_proxy] = median_value

# TODO Step 12: re-score with the suspect feature ablated and report:
#   - the change in P(approve) for each case (Δ before → after)
#   - whether the decision flipped (was DECLINE, now APPROVE)
#   - the new top-3 |SHAP| features under the ablation, vs before

pass

## 6. Audit Memo (fill in below)

> **Audit Memo — UdaciBank Loan-Approval Model — Cases CASE-001 and CASE-002.**
>
> **Model behavior summary:**  [Your response here — what did the model do on both cases?]
>
> **Explanation evidence:**
> - **CASE-001** — top features (sign): [Your response here — list the top 5 from your SHAP analysis]. See `shap_explanations_case1.png`.
> - **CASE-002** — top features (sign): [Your response here — list the top 5 from your SHAP analysis]. See `shap_explanations_case2.png`.
>
> **Counterfactual finding:** [Your response here — which proxy did you ablate, what happened to P(approve), did the decision flip?]
>
> **Proxy-attribute findings:** [Your response here — which features did `detect_proxy_features()` flag, and what protected attribute does each one proxy for?]
>
> **Caveats on the limits of SHAP** *(include this caveat sentence verbatim or paraphrased)*: SHAP shows correlation between features and model output, not causation; values are local and depend on the background distribution; class-imbalance can amplify low-frequency-feature attribution.
>
> **Recommended next steps per case** *(pick from the menu — overturn / re-review / escalate to fairness audit / decline-stand)*:
> - **CASE-001:** [pick from menu, with one-sentence rationale]
> - **CASE-002:** [pick from menu, with one-sentence rationale]
> - **Model-level:** [pick from menu — recommendation on the model retrain (pull suspect features, run fairness audit, escalate to AIRB), with one-sentence rationale]